In [ ]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, row_number, rank, dense_rank,
    lag, lead, sum, avg, count
)
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Week2_Day3").master("local[*]").getOrCreate()

emp_data = [
    (1, "Alice",   "Engineering", 95000),
    (2, "Bob",     "Engineering", 90000),
    (3, "Charlie", "Engineering", 90000),
    (4, "Diana",   "Data",        85000),
    (5, "Eve",     "Data",        80000),
    (6, "Frank",   "Data",        80000),
    (7, "Grace",   "HR",          70000),
    (8, "Hank",    "HR",          65000),
]

emp_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("department", StringType(), False),
    StructField("salary", IntegerType(), False),
])

df_emp = spark.createDataFrame(emp_data, emp_schema)

# Monthly sales data (for running totals and lag/lead)
sales_data = [
    ("North", "Jan", 10000), ("North", "Feb", 15000),
    ("North", "Mar", 12000), ("North", "Apr", 18000),
    ("South", "Jan", 8000),  ("South", "Feb", 9000),
    ("South", "Mar", 11000), ("South", "Apr", 7000),
]

sales_schema = StructType([
    StructField("region", StringType(), False),
    StructField("month", StringType(), False),
    StructField("revenue", IntegerType(), False),
])

df_sales = spark.createDataFrame(sales_data, sales_schema)

In [ ]:
# "Within each department, order employees by salary descending"
window_dept = Window.partitionBy("department").orderBy(col("salary").desc())

In [ ]:
df_ranked = df_emp.withColumn(
    "row_num", 
    row_number().over(window_dept)
)

df_ranked.show()

In [ ]:
df_ranked2 = df_emp.withColumn(
    "rank", 
    rank().over(window_dept)
)

df_ranked2.show()

In [ ]:
window_monthly = Window.partitionBy("region").orderBy("month")

df_mon = df_sales.withColumn(
    "prev_month_revenue",
    lag("revenue",1).over(window_monthly)
)
df_mon.show()

In [ ]:
df_change = df_mon.withColumn(
    "change",
    col("revenue")-col("prev_month_revenue")
)

df_change.show()

In [ ]:
window_dept = Window.partitionBy("department").orderBy(col("salary").desc())

df_full = (
    df_emp
    .withColumn("rank", row_number().over(window_dept))
    .withColumn("dept_max_salary", max("salary").over(Window.partitionBy("department")))
    .withColumn("salary_gap_from_top", col("dept_max_salary") - col("salary"))
)

df_full.show()